In [1]:
from pathlib import Path
import sys

from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/pratik0620/adaptive-federated-aggregation.git"
REPO_ROOT = Path("/content/adaptive-federated-aggregation")
if not REPO_ROOT.exists():
    !git clone {REPO_URL} {REPO_ROOT}

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SRC_DIR = REPO_ROOT / "src"
if not SRC_DIR.exists():
    raise FileNotFoundError(f"src directory not found at {SRC_DIR}")

print("Repository:", REPO_ROOT)
print("Source modules:", SRC_DIR)

Mounted at /content/drive
Cloning into '/content/adaptive-federated-aggregation'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 36 (delta 17), reused 34 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 19.76 KiB | 3.29 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Repository: /content/adaptive-federated-aggregation
Source modules: /content/adaptive-federated-aggregation/src


In [2]:
# Install the packages required by this notebook and the shared preprocessing utilities.
%pip install -q numpy pandas scikit-learn matplotlib jupyter


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 71.2 MB/s eta 0:00:00


## Configuration — Google Drive Paths

The next code cell defines the only machine-specific setting, `BASE_DRIVE_PATH`, and validates the three NIH metadata files. This notebook does not require an image directory and does not download or copy X-ray images.

# NIH ChestX-ray14 preprocessing

This notebook creates the NIH federated client as metadata-only CSV files. It keeps clean target positives and explicit `No Finding` negatives, excludes ambiguous label combinations, preserves patient IDs, and splits whole patients. Images are never copied or downloaded in this phase.

**Storage model:** Code and notebooks are stored in GitHub, while the large NIH metadata files and future image files are stored in Google Drive. The actual X-ray images will be downloaded later, after the target disease and required sample size are finalized.

In [3]:
import json, platform
import numpy as np
import pandas as pd

from src.config import TARGET_DISEASE, RANDOM_SEED, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES
from src.dataset_utils import nih_clean_binary_labels, standardize_nih
from src.preprocessing_utils import (assert_no_leakage, assert_no_study_leakage, build_summary,
                                     require_file, sample_clean_cases, save_splits,
                                     save_summary, set_reproducibility, split_by_patient)

set_reproducibility(RANDOM_SEED)
print('Python:', platform.python_version())
print('pandas:', pd.__version__, 'numpy:', np.__version__)

# Google Drive configuration for raw NIH metadata and metadata-only outputs.
BASE_DRIVE_PATH = Path("/content/drive/MyDrive/Healthcare_FL")
NIH_ROOT = BASE_DRIVE_PATH / "raw" / "NIH"
NIH_METADATA_DIR = NIH_ROOT / "metadata"

METADATA_FILE = NIH_METADATA_DIR / "Data_Entry_2017.csv"
TRAIN_VAL_FILE = NIH_METADATA_DIR / "train_val_list.txt"
TEST_FILE = NIH_METADATA_DIR / "test_list.txt"
OUTPUT_DIR = BASE_DRIVE_PATH / "processed" / "client_01_NIH"

required_files = [METADATA_FILE, TRAIN_VAL_FILE, TEST_FILE]
for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Required NIH dataset file not found: {file_path}")

metadata_path = require_file(METADATA_FILE, "NIH metadata CSV")
train_val_path = require_file(TRAIN_VAL_FILE, "NIH train/validation split list")
test_path = require_file(TEST_FILE, "NIH test split list")

print("Repository:", REPO_ROOT)
print("Google Drive:", BASE_DRIVE_PATH)
print("Dataset:", NIH_ROOT)
print("Metadata:", METADATA_FILE)
print("NIH train/validation split path:", TRAIN_VAL_FILE)
print("NIH test split path:", TEST_FILE)
print("All required NIH metadata files exist.")
print("Target disease:", TARGET_DISEASE)

Python: 3.13.15
pandas: 2.2.3 numpy: 2.1.3
Repository: /content/adaptive-federated-aggregation
Google Drive: /content/drive/MyDrive/Healthcare_FL
Dataset: /content/drive/MyDrive/Healthcare_FL/raw/NIH
Metadata: /content/drive/MyDrive/Healthcare_FL/raw/NIH/metadata/Data_Entry_2017.csv
NIH train/validation split path: /content/drive/MyDrive/Healthcare_FL/raw/NIH/metadata/train_val_list.txt
NIH test split path: /content/drive/MyDrive/Healthcare_FL/raw/NIH/metadata/test_list.txt
All required NIH metadata files exist.
Target disease: Pleural Effusion


## Load and inspect
NIH `Finding Labels` is pipe-separated. The clean binary policy is deliberately conservative: a target-containing study is positive; exactly `No Finding` is negative; every other combination is excluded.

In [4]:
nih = pd.read_csv(metadata_path)
train_val_ids = pd.read_csv(train_val_path, header=None, names=['image_id'])
test_ids = pd.read_csv(test_path, header=None, names=['image_id'])

print('images:', len(nih))
print('unique patients:', nih['Patient ID'].nunique())
print('columns:', nih.columns.tolist())
print('official train/validation image IDs:', len(train_val_ids))
print('official test image IDs:', len(test_ids))
print('official split image overlap:', len(set(train_val_ids['image_id']) & set(test_ids['image_id'])))
display(nih.isna().sum().sort_values(ascending=False).head(20).to_frame('missing'))
display(nih['Finding Labels'].value_counts(dropna=False).head(20).to_frame('label_count'))
clean, counts = nih_clean_binary_labels(nih, TARGET_DISEASE)
print('Counts before filtering:', counts)
print('Clean rows:', len(clean))

images: 112120
unique patients: 30805
columns: ['Image Index', 'Finding Labels', 'Follow-up #', 'Patient ID', 'Patient Age', 'Patient Gender', 'View Position', 'OriginalImage[Width', 'Height]', 'OriginalImagePixelSpacing[x', 'y]', 'Unnamed: 11']
official train/validation image IDs: 86524
official test image IDs: 25596
official split image overlap: 0


,missing
Unnamed: 11,112120
Image Index,0
Follow-up #,0
Finding Labels,0
Patient ID,0
Patient Age,0
View Position,0
Patient Gender,0
OriginalImage[Width,0
Height],0


,label_count
Finding Labels,
No Finding,60361
Infiltration,9547
Atelectasis,4215
Effusion,3955
Nodule,2705
Pneumothorax,2194
Mass,2139
Effusion|Infiltration,1603
Atelectasis|Infiltration,1350


Counts before filtering: {'positive_before_filter': 0, 'negative_before_filter': 60361, 'excluded_ambiguous_before_filter': 51759}
Clean rows: 60361


In [5]:
# No image directory is required for this metadata-only phase.
standardized = standardize_nih(clean, image_dir=None)
standardized = sample_clean_cases(standardized, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES, RANDOM_SEED)
print('Rows after optional sampling:', len(standardized))
display(standardized.head())
print(standardized['target'].value_counts().sort_index())
splits = split_by_patient(standardized, seed=RANDOM_SEED)
assert_no_leakage(splits)
assert_no_study_leakage(splits)
print('patient leakage checks: passed')
print('study leakage checks: passed where study_id exists')
save_splits(splits, OUTPUT_DIR)
summary = build_summary(splits, 'NIH', {'target_disease': TARGET_DISEASE, 'random_seed': RANDOM_SEED, 'max_positive_samples': MAX_POSITIVE_SAMPLES, 'max_negative_samples': MAX_NEGATIVE_SAMPLES, 'filter_counts_before_filtering': counts, 'official_split_lists_loaded': True, 'official_split_used': False, 'image_paths_materialized': False})
save_summary(summary, OUTPUT_DIR)
print(json.dumps(summary, indent=2))
print('Saved:', sorted(path.name for path in OUTPUT_DIR.glob('*')))

Rows after optional sampling: 60361


,image_id,patient_id,study_id,source,target,age,sex,view_position,projection,image_width,image_height,pixel_spacing_x,pixel_spacing_y,scanner_manufacturer,image_path,finding_labels,follow_up
0,00006763_000.png,6763,NaN,NIH,0,13,M,PA,PA,NaN,NaN,NaN,NaN,NaN,NaN,No Finding,0
1,00024141_000.png,24141,NaN,NIH,0,26,M,PA,PA,NaN,NaN,NaN,NaN,NaN,NaN,No Finding,0
2,00028639_001.png,28639,NaN,NIH,0,28,F,PA,PA,NaN,NaN,NaN,NaN,NaN,NaN,No Finding,1
3,00014768_004.png,14768,NaN,NIH,0,11,M,AP,AP,NaN,NaN,NaN,NaN,NaN,NaN,No Finding,4
4,00026243_000.png,26243,NaN,NIH,0,66,M,PA,PA,NaN,NaN,NaN,NaN,NaN,NaN,No Finding,0


target
0    60361
Name: count, dtype: int64
patient leakage checks: passed
study leakage checks: passed where study_id exists
{
  "source": "NIH",
  "parameters": {
    "target_disease": "Pleural Effusion",
    "random_seed": 42,
    "max_positive_samples": null,
    "max_negative_samples": null,
    "filter_counts_before_filtering": {
      "positive_before_filter": 0,
      "negative_before_filter": 60361,
      "excluded_ambiguous_before_filter": 51759
    },
    "official_split_lists_loaded": true,
    "official_split_used": false,
    "image_paths_materialized": false
  },
  "splits": {
    "train": {
      "images": 48373,
      "patients": 19925,
      "studies": 0,
      "positive": 0,
      "negative": 48373
    },
    "val": {
      "images": 5841,
      "patients": 2491,
      "studies": 0,
      "positive": 0,
      "negative": 5841
    },
    "test": {
      "images": 6147,
      "patients": 2491,
      "studies": 0,
      "positive": 0,
      "negative": 6147
    }
  },
 